In [2]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_pseudobulk_de"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_pseudobulk_de"

# ----------------------------
# M1/M2 polarization scoring — direct extension of Azizi et al. 2018's
# finding that M1/M2 marker genes are co-expressed in the same cells,
# tested here at single-cell resolution using our own validated
# macrophage sub-clusters (now confirmed correct after today's fixes).
# Also checks where the tumour-vs-normal reprogramming genes (FN1,
# HSPA1A/B) sit relative to this axis.
# ----------------------------
adata1_mac = sc.read_h5ad(PROCESSED_DIR / "GSE114725_macrophages_subclustered.h5ad")
print(f"Loaded: {adata1_mac.n_obs} macrophages")

m1_genes = ["IL1B", "TNF", "NOS2", "CD86", "CXCL9", "CXCL10"]
m2_genes = ["CD163", "MRC1", "IL10", "ARG1", "CD206", "MSR1"]

mac_raw = adata1_mac.raw.to_adata()
mac_raw.obs["mac_subtype"] = adata1_mac.obs["mac_subtype"].values

m1_present = [g for g in m1_genes if g in mac_raw.var_names]
m2_present = [g for g in m2_genes if g in mac_raw.var_names]
print(f"M1 markers found: {m1_present}")
print(f"M2 markers found: {m2_present}")

sc.tl.score_genes(mac_raw, m1_present, score_name="M1_score")
sc.tl.score_genes(mac_raw, m2_present, score_name="M2_score")

r, p = pearsonr(mac_raw.obs["M1_score"], mac_raw.obs["M2_score"])
print(f"\nM1 vs M2 score correlation across all macrophages: r={r:.3f}, p={p:.2e}")

# Per sub-cluster breakdown
print("\nM1/M2 scores by sub-cluster:")
print(mac_raw.obs.groupby("mac_subtype", observed=True)[["M1_score", "M2_score"]].mean().round(3))

# Scatter plot, coloured by sub-cluster
fig, ax = plt.subplots(figsize=(8, 7))
for subtype in mac_raw.obs["mac_subtype"].unique():
    mask = mac_raw.obs["mac_subtype"] == subtype
    ax.scatter(mac_raw.obs.loc[mask, "M1_score"], mac_raw.obs.loc[mask, "M2_score"],
               label=subtype, alpha=0.5, s=10)
ax.set_xlabel("M1 score")
ax.set_ylabel("M2 score")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
ax.set_title(f"M1 vs M2 polarization score, per cell (r={r:.2f})")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "GSE114725_macrophage_M1_M2_spectrum.png", dpi=300, bbox_inches="tight")
plt.close()

# Where do the reprogramming genes sit relative to this axis?
print("\nReprogramming genes (FN1, HSPA1A/B) vs M1/M2 axis:")
for gene in ["FN1", "HSPA1A", "HSPA1B"]:
    if gene in mac_raw.var_names:
        X = mac_raw[:, gene].X
        if hasattr(X, "toarray"): X = X.toarray()
        r_m1, _ = pearsonr(X.flatten(), mac_raw.obs["M1_score"])
        r_m2, _ = pearsonr(X.flatten(), mac_raw.obs["M2_score"])
        print(f"  {gene}: correlation with M1={r_m1:.3f}, M2={r_m2:.3f}")

print("\nSaved: GSE114725_macrophage_M1_M2_spectrum.png")

Loaded: 5914 macrophages
M1 markers found: ['IL1B', 'TNF', 'CD86', 'CXCL9', 'CXCL10']
M2 markers found: ['CD163', 'MRC1', 'IL10', 'MSR1']

M1 vs M2 score correlation across all macrophages: r=0.007, p=5.95e-01

M1/M2 scores by sub-cluster:
                                                    M1_score  M2_score
mac_subtype                                                           
LAM-like macrophages                                  -0.027    -0.174
Lipid-laden/Foam-cell macrophages                     -0.108    -0.182
Antigen-presenting macrophages                         0.083    -0.158
Complement-high macrophages                            0.044    -0.071
Resting/Resident macrophages                          -0.044     0.571
Monocyte-like macrophages                              0.062    -0.167
Non-classical monocytes (CD16+)                       -0.025    -0.538
Unassigned (n=91, stromal/RBC contamination art...    -0.110    -0.258

Reprogramming genes (FN1, HSPA1A/B) vs M1/M2 axis

In [4]:
from scipy.stats import spearmanr

In [5]:
# ----------------------------
# NEW — Per-patient Treg vs macrophage M2-skew correlation, per
# Adrien's suggestion (30/07 meeting): does Treg abundance relate to
# which patients show more M2-skewed macrophages? A different question
# from the single-cell M1/M2 correlation above (which found nothing) —
# this tests a patient-level relationship between two related
# immunosuppressive populations.
# ----------------------------
adata1_full = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")

# Per-patient Treg proportion (of total cells)
patient_totals = adata1_full.obs.groupby("patient", observed=True).size()
treg_counts = adata1_full.obs[
    adata1_full.obs["cell_type_fine"] == "Regulatory T cells (Tregs, CD4+)"
].groupby("patient", observed=True).size()
treg_pct = (treg_counts / patient_totals * 100).reindex(patient_totals.index, fill_value=0)

print("Treg % per patient:")
print(treg_pct.sort_values(ascending=False))

# Per-patient mean M2 score, restricted to macrophages (using mac_raw
# and M2_score already computed above in this notebook)
mac_raw.obs["patient"] = mac_raw.obs["patient"].astype(str)
patient_m2 = mac_raw.obs.groupby("patient", observed=True)["M2_score"].mean()

print("\nMean macrophage M2 score per patient:")
print(patient_m2.sort_values(ascending=False))

# Align and correlate — only patients present in both
common_patients = treg_pct.index.intersection(patient_m2.index)
print(f"\nPatients with both measures: {len(common_patients)}")

if len(common_patients) >= 4:
    r, p = pearsonr(treg_pct.loc[common_patients], patient_m2.loc[common_patients])
    print(f"\nTreg % vs mean macrophage M2 score (Pearson): r={r:.3f}, p={p:.3f}")
    rho, p_s = spearmanr(treg_pct.loc[common_patients], patient_m2.loc[common_patients])
    print(f"Treg % vs mean macrophage M2 score (Spearman): rho={rho:.3f}, p={p_s:.3f}")
else:
    print("Too few patients with both measures for a meaningful correlation")

Treg % per patient:
patient
BC5    11.111111
BC2     9.555149
BC4     7.067249
BC1     4.932866
BC7     4.562534
BC8     4.198767
BC6     3.545896
BC3     1.562500
dtype: float64

Mean macrophage M2 score per patient:
patient
BC3    0.202460
BC5   -0.033419
BC6   -0.050837
BC2   -0.097109
BC4   -0.182742
BC8   -0.239813
BC7   -0.266991
BC1   -0.292404
Name: M2_score, dtype: float64

Patients with both measures: 8

Treg % vs mean macrophage M2 score (Pearson): r=-0.146, p=0.731
Treg % vs mean macrophage M2 score (Spearman): rho=-0.119, p=0.779


In [6]:
tumor_mask = adata1_full.obs["tissue"] == "TUMOR"
patient_totals_tumor = adata1_full.obs[tumor_mask].groupby("patient", observed=True).size()
treg_counts_tumor = adata1_full.obs[
    tumor_mask & (adata1_full.obs["cell_type_fine"] == "Regulatory T cells (Tregs, CD4+)")
].groupby("patient", observed=True).size()
treg_pct_tumor = (treg_counts_tumor / patient_totals_tumor * 100).reindex(patient_totals_tumor.index, fill_value=0)

mac_tumor_mask = mac_raw.obs["tissue"] == "TUMOR"
patient_m2_tumor = mac_raw.obs[mac_tumor_mask].groupby("patient", observed=True)["M2_score"].mean()

common_tumor = treg_pct_tumor.index.intersection(patient_m2_tumor.index)
print(f"Patients with Tumour-only data for both measures: {len(common_tumor)}")
r_t, p_t = pearsonr(treg_pct_tumor.loc[common_tumor], patient_m2_tumor.loc[common_tumor])
print(f"Tumour-only: Treg % vs macrophage M2 score: r={r_t:.3f}, p={p_t:.3f}")

Patients with Tumour-only data for both measures: 8
Tumour-only: Treg % vs macrophage M2 score: r=-0.212, p=0.614


In [7]:
tumor_mask = mac_raw.obs["tissue"] == "TUMOR"
normal_mask = mac_raw.obs["tissue"] == "NORMAL"

print(f"Tumour macrophages: mean M2 score = {mac_raw.obs.loc[tumor_mask, 'M2_score'].mean():.3f} (n={tumor_mask.sum()})")
print(f"Normal macrophages: mean M2 score = {mac_raw.obs.loc[normal_mask, 'M2_score'].mean():.3f} (n={normal_mask.sum()})")

from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(mac_raw.obs.loc[tumor_mask, "M2_score"], mac_raw.obs.loc[normal_mask, "M2_score"])
print(f"Mann-Whitney U test: p={p:.4f}")

Tumour macrophages: mean M2 score = -0.094 (n=4969)
Normal macrophages: mean M2 score = 0.027 (n=445)
Mann-Whitney U test: p=0.0004


In [8]:
tumor_mask = mac_raw.obs["tissue"] == "TUMOR"
normal_mask = mac_raw.obs["tissue"] == "NORMAL"

print(f"Tumour macrophages: mean M1 score = {mac_raw.obs.loc[tumor_mask, 'M1_score'].mean():.3f} (n={tumor_mask.sum()})")
print(f"Normal macrophages: mean M1 score = {mac_raw.obs.loc[normal_mask, 'M1_score'].mean():.3f} (n={normal_mask.sum()})")

from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(mac_raw.obs.loc[tumor_mask, "M1_score"], mac_raw.obs.loc[normal_mask, "M1_score"])
print(f"Mann-Whitney U test: p={p:.4f}")

Tumour macrophages: mean M1 score = 0.017 (n=4969)
Normal macrophages: mean M1 score = -0.011 (n=445)
Mann-Whitney U test: p=0.1673


In [9]:
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from scipy.stats import pearsonr

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"

adata1_mac = sc.read_h5ad(PROCESSED_DIR / "GSE114725_macrophages_subclustered.h5ad")
mac_raw = adata1_mac.raw.to_adata()
mac_raw.obs["mac_subtype"] = adata1_mac.obs["mac_subtype"].values
mac_raw.obs["patient"] = adata1_mac.obs["patient"].astype(str).values
mac_raw.obs["tissue"] = adata1_mac.obs["tissue"].values

# ----------------------------
# Extended panel — M1 and four M2 sub-states (M2a/b/c/d), per the
# supplied reference table. Only clear, unambiguous gene symbols used;
# protein complexes (MHC II), receptors without a single clear gene
# symbol (SR, DecoyR), and ambiguous abbreviations (FIZZ1 -> RETNLB)
# excluded or resolved explicitly below.
# ----------------------------
panels = {
    "M1": ["CD86", "CD80", "CD68", "TLR1", "TLR2", "TLR4", "NOS2", "SOCS3",
           "TNF", "IL1B", "IL6", "IL12A", "IL12B", "IL23A",
           "CCL5", "CCL8", "CCL2", "CCL3", "CCL4"],
    "M2a": ["CD163", "MRC1", "CD200R1",
            "IL10", "TGFB1",
            "CCL17", "CCL22", "CCL24", "CCL13", "CCL18"],
    "M2b": ["CD86",
            "IL1B", "IL6", "TNF", "IL10", "IL12A", "IL12B",
            "CCL1"],
    "M2c": ["CD163", "MRC1",
            "IL10", "TGFB1",
            "CXCL10", "CXCL16"],
    "M2d": ["VEGFA", "IL10RA", "ARG1",
            "IL10", "TNF", "TGFB1",
            "CCL5"],
}

results = {}
for name, genes in panels.items():
    present = [g for g in genes if g in mac_raw.var_names]
    missing = [g for g in genes if g not in mac_raw.var_names]
    print(f"{name}: {len(present)}/{len(genes)} genes found. Missing: {missing}")
    sc.tl.score_genes(mac_raw, present, score_name=f"{name}_score")
    results[name] = present

print("\n=== Pairwise correlations between all 5 states ===")
score_cols = [f"{name}_score" for name in panels.keys()]
for a, b in combinations(score_cols, 2):
    r, p = pearsonr(mac_raw.obs[a], mac_raw.obs[b])
    print(f"  {a} vs {b}: r={r:.3f}, p={p:.2e}")

print("\n=== Mean score by sub-cluster ===")
print(mac_raw.obs.groupby("mac_subtype", observed=True)[score_cols].mean().round(3))

M1: 16/19 genes found. Missing: ['NOS2', 'IL12A', 'IL12B']
M2a: 9/10 genes found. Missing: ['CCL24']
M2b: 5/8 genes found. Missing: ['IL12A', 'IL12B', 'CCL1']
M2c: 6/6 genes found. Missing: []
M2d: 6/7 genes found. Missing: ['ARG1']

=== Pairwise correlations between all 5 states ===
  M1_score vs M2a_score: r=0.142, p=4.86e-28
  M1_score vs M2b_score: r=0.402, p=4.94e-229
  M1_score vs M2c_score: r=0.135, p=2.24e-25
  M1_score vs M2d_score: r=0.259, p=3.12e-91
  M2a_score vs M2b_score: r=0.052, p=6.65e-05
  M2a_score vs M2c_score: r=0.768, p=0.00e+00
  M2a_score vs M2d_score: r=0.204, p=2.47e-56
  M2b_score vs M2c_score: r=0.107, p=1.23e-16
  M2b_score vs M2d_score: r=0.235, p=7.33e-75
  M2c_score vs M2d_score: r=0.251, p=6.58e-86

=== Mean score by sub-cluster ===
                                                    M1_score  M2a_score  \
mac_subtype                                                               
LAM-like macrophages                                   0.163     -0.048  

In [10]:
from scipy.stats import mannwhitneyu

tumor_mask = mac_raw.obs["tissue"] == "TUMOR"
normal_mask = mac_raw.obs["tissue"] == "NORMAL"

for score in ["M2a_score", "M2c_score", "M1_score", "M2b_score", "M2d_score"]:
    tumor_mean = mac_raw.obs.loc[tumor_mask, score].mean()
    normal_mean = mac_raw.obs.loc[normal_mask, score].mean()
    stat, p = mannwhitneyu(mac_raw.obs.loc[tumor_mask, score], mac_raw.obs.loc[normal_mask, score])
    print(f"{score}: Tumour={tumor_mean:.3f}, Normal={normal_mean:.3f}, p={p:.4f}")

M2a_score: Tumour=-0.013, Normal=0.007, p=0.5006
M2c_score: Tumour=0.078, Normal=0.183, p=0.0000
M1_score: Tumour=0.191, Normal=0.081, p=0.0000
M2b_score: Tumour=0.019, Normal=-0.011, p=0.0474
M2d_score: Tumour=0.086, Normal=0.002, p=0.0000


In [11]:
for gene in ["FN1", "HSPA1A", "HSPA1B"]:
    X = mac_raw[:, gene].X
    if hasattr(X, "toarray"): X = X.toarray()
    r, p = pearsonr(X.flatten(), mac_raw.obs["M1_score"])
    print(f"{gene} vs broad M1_score: r={r:.3f}, p={p:.2e}")

FN1 vs broad M1_score: r=0.198, p=4.27e-53
HSPA1A vs broad M1_score: r=0.173, p=9.67e-41
HSPA1B vs broad M1_score: r=0.143, p=3.27e-28
